In [1]:
import CardScraperJP
import warnings
warnings.filterwarnings('ignore')

# start: 33001, end: 38493
start = 33299
end = 38493
CardScraperJP.scrapeCards(start, end)

33299
33300
33301
33302
33303
33304
33305
33306
33307
33308
33309
33310
33311
33312
33313
33314
33315
33316
33317
33318
33319
33320
33321
33322
33323
33324
33325
33326
33327
33328
33329
33330
33331
33332
33333
33334
33335
33336
33337
33338
33339
33340
33341
33342
33343
33344
33345
33346
33347
33348
33349
33350
33351
33352
33353
33354
33355
33356
33357
33358
33359
33360
33361
33362
33363
33364
33365
33366
33367
33368
33369
33370
33371
33372
33373
33374
33375
33376
33377
33378
33379
33380
33381
33382
33383
33384
33385
33386
33387
33388
33389
33390
33391
33392
33393
33394
33395
33396
33397
33398
33399
33400
33401
33402
33403
33404
33405
33406
33407
33408
33409
33410
33411
33412
33413
33414
33415
33416
33417
33418
33419
33420
33421
33422
33423
33424
33425
33426
33427
33428
33429
33430
33431
33432
33433
33434
33435
33436
33437
33438
33439
33440
33441
33442
33443
33444
33445
33446
33447
33448
33449
33450
33451
33452
33453
33454
33455
33456
33457
33458
33459
33460
33461
33462
33463
33464
3346

IndexError: list index out of range

In [3]:
import bs4
import requests


In [4]:
def readEnergy(p):
    spans = p.find_all('span')
    if spans:
        energies = [span['class'][0].split('-')[1] for span in spans]
        for i in range(len(energies)):
            p = str(p).replace(str(spans[i]), energies[i])
        p = bs4.BeautifulSoup(p)
    p = p.get_text().replace('\n   ', '')
    return p

In [5]:
def readName(h1):
    span = h1.find('span')
    if span:
        prismstar = span['class'][1].split('-')[1]
        h1 = str(h1).replace(str(span), prismstar)
        h1 = bs4.BeautifulSoup(h1)
    h1 = h1.get_text().replace('\n   ', '')
    return h1

In [6]:
def getContent(url):
    # anti-scraping
    user_agent = "Mozilla/5.0 (Windows NT 10.0; WOW64; rv:68.0) Gecko/20100101 Firefox/68.0"
    url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
    response = requests.get(url, headers={'User-Agent': user_agent})
    if response.status_code == 200:
        # print(response.content.decode('utf-8'))
        return response.content.decode('utf-8')
    else:
        print(f"Fail to get the url [{response.status_code}]")

In [7]:
def readCard(cardId):
    # anti-scraping
    user_agent = "Mozilla/5.0 (Windows NT 10.0; WOW64; rv:68.0) Gecko/20100101 Firefox/68.0"
    url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
    response = requests.get(url, headers={'User-Agent': user_agent})
    if response.status_code == 200:
        # print(response.content.decode('utf-8'))
        content = response.content.decode('utf-8')
    else:
        print(f"Fail to get the url [{response.status_code}]")
        
    # start reading content
    soup = bs4.BeautifulSoup(content, 'html.parser')
    card = soup.section
    # all type cards
    name = readName(card.h1)
    img = card.find('img', class_='fit')['src']
    
    # init
    [reg,         setNum,    setCount,   rarity,      dexNum,    dexClass,
     height,      weight,    dexDesc,    author,      desc,      stage, 
     hp,          pType,     ability,    abilityDesc, waza1Cost, waza1Name,
     waza1Damage, waza1Desc, waza2Cost,  waza2Name,   waza2Damage, waza2Desc,
     GXCost, GXName, GXDamage, GXDesc,
     weakType,    weakValue, resistType, resistValue, escape, spRule] = [''] * 34
    
    # decide card type
    cardType = card.h2.get_text()
    if cardType == '基本エネルギー':
        return [cardId, cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
            GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]


    ## left box
    reg = card.find('img', class_='img-regulation')['alt'] # regulation
    regImg = card.find('img', class_='img-regulation')['src']
    
    setInfo = card.find('div', class_='subtext').get_text().strip()
    if len(setInfo.split('/')) == 2:
        setCount = setInfo.strip()[-3:]
        if setCount.isdigit():
            setCount = int(setCount)
        setNum = int(setInfo.strip()[:3])
    else:
        setCount = setInfo

    if card.find('img', width='24'):
        rarityImg = card.find('img', width='24')['src']
        rarity = rarityImg.split('.')[0].split('rare_')[1]
    
    if cardType == '特殊エネルギー':
        desc = readEnergy(card.find('p'))
        return [cardId, cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
                GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]

    author = card.find('div', class_='author').get_text().strip().split('\n')[1]

    ### global pokedex
    pokedex = card.find('div', class_='card')
    if pokedex: # has pokedex
        if pokedex.h4:
            [dexNum, dexClass] = pokedex.h4.get_text().strip().split('\u3000')
            dexNum = int(dexNum.split('.')[1])
        if len(pokedex.find_all('p')) == 2:
            htAndWt = pokedex.p.get_text().split('：')
            height = float(htAndWt[1].split(' ')[0])
            weight = float(htAndWt[2].split(' ')[0])
            dexDesc = pokedex.find_all('p')[1].get_text()
        else:
            dexDesc = pokedex.find('p').get_text()
    
    if cardType in ['サポート', 'グッズ', 'ポケモンのどうぐ', 'スタジアム']:
        desc = card.find_all('p')
        if cardType in ['サポート', 'グッズ', 'スタジアム']:
            desc = readEnergy(desc[0])
        if cardType == 'ポケモンのどうぐ':
            desc = readEnergy(desc[1])
        return [cardId, cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
                GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]
    
    cardType = 'pokemon'

    ## right box
    stage = card.find('span', class_='type').get_text()
    if '\xa0' in stage:
        stage = stage.replace('\xa0', ' ')
    hp = card.find('span', class_='hp-num').get_text()
    hp = int(hp)
    pType = card.find('div', class_='td-r').find_all('span')[-1]['class'][0].split('-')[1]
    
    ### waza part
    part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
    soup = bs4.BeautifulSoup(part)
    wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))
    
    h2 = wazaPart.find_all('h2')
    skills = wazaPart.find_all('h4')
    p = wazaPart.find_all('p')
    
    for area in h2:
        areaType = area.get_text().strip()
        if areaType == "特性":
            # ability
            # print('learning an ability')
            ability = skills[0].get_text().strip()
            abilityDesc = readEnergy(p[0]).strip()
            del skills[0]

        elif areaType == "特別なルール":
            # special rule
            # print('learning a special rule')
            spRule = readName(p[-1]).strip()
            del p[-1]
            
        elif areaType == "GXワザ":
            # GX waza
            [GXCost, GXName, GXDamage, GXDesc] = [''] * 4
            # print('learning a GX attack')
            GXCost = [span['class'][0].split('-')[1] for span in skills[-1].find_all('span', class_='icon')]
            GX = skills[-1].get_text().strip().split(' ')
            GXName = GX[0].strip()
            GXDamage = GX[-1]
            if not GXDamage[-2].isdigit():
                GXDamage = ''
            GXDesc = skills[-1].find_next_sibling('p')
            GXDesc = readEnergy(GXDesc).strip()
            del skills[-1], p[-2]
        
    # waza
    waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
    waza1 = skills[0].get_text().strip().split(' ')
    waza1Name = waza1[0].strip()
    waza1Damage = waza1[-1]
    if not waza1Damage[-2].isdigit():
        waza1Damage = ''
    waza1Desc = skills[0].find_next_sibling('p')
    waza1Desc = readEnergy(waza1Desc).strip()

    if len(skills) > 1:
        waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
        waza2 = skills[1].get_text().strip().split(' ')
        waza2Name = waza2[0].strip()
        waza2Damage = waza2[-1]
        if not waza2Damage[-2].isdigit():
            waza2Damage = ''
        waza2Desc = skills[1].find_next_sibling('p')
        waza2Desc = readEnergy(waza2Desc).strip()
    
    ### table
    td = wazaPart.find_all('td')
    if td[0].find('span'):
        weakType = td[0].find('span')['class'][0].split('-')[1]
        weakValue = td[0].get_text().strip()

    if td[1].find('span'):
        resistType = td[1].find('span')['class'][0].split('-')[1]
        resistValue = td[1].get_text().strip()

    escape = len(td[2].find_all('span'))
    

    return [cardId, cardType, name, img, reg, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, dexDesc, author, desc, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
            GXCost, GXName, GXDamage, GXDesc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]

In [8]:
# read pokemon card
# url = "https://www.pokemon-card.com/card-search/details.php/card/38407/regu/XY" # biidoru c
# url = "https://www.pokemon-card.com/card-search/details.php/card/38439/regu/XY" # zacian as
# url = "https://www.pokemon-card.com/card-search/details.php/card/38438/regu/XY" # mahoippu Vmax
# url = "https://www.pokemon-card.com/card-search/details.php/card/38426/regu/XY" # zekuromu r
# url = "https://www.pokemon-card.com/card-search/details.php/card/38419/regu/XY" # zaruudo V 38493
# url = "https://www.pokemon-card.com/card-search/details.php/card/38423/regu/XY" # raiboruto c ability
# url = "https://www.pokemon-card.com/card-search/details.php/card/38310/regu/XY" # mew V resistance
# url = "https://www.pokemon-card.com/card-search/details.php/card/38300/regu/XY" # pikachu V
# url = "https://www.pokemon-card.com/card-search/details.php/card/38304/regu/XY" # denryu


cardId = 33580 # start: 33001, end: 38493
# url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
# content = getContent(url)
readCard(cardId)

IndexError: list index out of range

In [10]:
# read other card
cardId = 38407 # pokemon
cardId = 35898 # GX
# cardId = 37281 # prismstar
# cardId = 38474 # dougu ポケモンのどうぐ
# cardId = 38470 # guzzu グッズ
# cardId = 33300 # guzzu S-P
# cardId = 33207 # stadium
# cardId = 37281
# cardId = 38477 # supporter サポート
# cardId = 38480 # special energy 特殊エネルギー
# cardId = 37743 # basic energy 基本エネルギー
# cardId = 37392 # EX
# cardId = 37406 # Mega
# cardId = 37406 # Mega
cardId = 33580 # break (no waza)


url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}'
content = getContent(url)

soup = bs4.BeautifulSoup(content, 'html.parser')
card = soup.section
# all type cards
name = card.h1.get_text()
img = card.find('img', class_='fit')['src']

In [11]:
card

<section class="Section">
<h1 class="Heading1 mt20">ゲッコウガBREAK</h1>
<div class="Box">
<div class="LeftBox">
<img alt="ゲッコウガBREAK" class="fit" src="/assets/images/card_images/legend/XY/033580_P_GEKKOUGABREAK.jpg"/>
<div class="subtext Text-fjalla">
<img alt="XY" class="img-regulation" src="/assets/images/card/regulation_logo_1/XY.gif"/> 028 / 171                     </div>
<span> </span>
<div class="author">
<h4>イラストレーター</h4>
<a href="/card-search/index.php?regulation_illust=all&amp;illust=5ban Graphics">5ban Graphics</a> </div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<div class="TopInfo Text-fjalla">
<div class="tr">
<div class="td-l">
<span class="type">BREAK進化</span>
</div>
<div class="td-r">
<span class="hp">HP</span>
<span class="hp-num">170</span>
<span class="hp-type">タイプ</span>
<span class="icon-water icon"></span> </div>
</div>
</div>
<h2 class="mt20">特性</h2>
<h4>きょだいみずしゅりけん</h4>
<p>自分の手札から<span class="icon-water icon"></span>エネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹

In [12]:
[span['class'][1].split('-')[1] for span in card.h1.find_all('span')]

[]

In [13]:
pokedex = card.find('div', class_='card')
pokedex

In [14]:
p = card.find('p')
p

<p>自分の手札から<span class="icon-water icon"></span>エネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。</p>

In [15]:
p.get_text()

'自分の手札からエネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。'

In [16]:
readEnergy(p)

'自分の手札からwaterエネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。'

In [17]:
spans = p.find_all('span')
spans

[<span class="icon-water icon"></span>]

In [18]:
energies = [span['class'][0].split('-')[1] for span in spans]
energies

['water']

In [31]:

for i in range(len(energies)):
    p = str(p).replace(str(spans[0]), energies[0])
# p = bs4.BeautifulSoup(p)
# p = p.get_text().replace('\n   ', '')

p

'<p>おたがいのプレイヤーは、自分の番ごとに1回、自分のwaterポケモンと<span class="icon-electric icon"></span>ポケモン全員のHPを、それぞれ「30」回復してよい。</p>'

In [ ]:
# decide card type
cardType = card.h2.get_text()
if cardType == '基本エネルギー':
    return
elif cardType in ['特殊エネルギー', 'サポート', 'グッズ', 'ポケモンのどうぐ']:
    desc = card.find_all('p')
else:
    cardType = 'pokemon'

In [19]:
part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
soup = bs4.BeautifulSoup(part)
wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))

In [20]:

wazaPart

<html>
<body>
<span class="icon-water icon">
</span>
<h2 class="mt20">
   特性
  </h2>
<h4>
   きょだいみずしゅりけん
  </h4>
<p>
   自分の手札から
   <span class="icon-water icon">
</span>
   エネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。
  </p>
<h2 class="mt20">
   特別なルール
  </h2>
<p>
   BREAK進化する前のゲッコウガの「ワザ・特性・弱点・抵抗力・にげる」を引きつぐ。
  </p>
<table cellpadding="0" cellspacing="0">
<tr>
<th>
     弱点
    </th>
<th>
     抵抗力
    </th>
<th>
     にげる
    </th>
</tr>
<tr>
<td>
     --
    </td>
<td>
     --
    </td>
<td class="escape">
</td>
</tr>
</table>
</body>
</html>

In [21]:
h2 = wazaPart.find_all('h2')
h2

[<h2 class="mt20">
    特性
   </h2>,
 <h2 class="mt20">
    特別なルール
   </h2>]

In [22]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
skills

[<h4>
    きょだいみずしゅりけん
   </h4>]

In [23]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
p = wazaPart.find_all('p')
p

[<p>
    自分の手札から
    <span class="icon-water icon">
 </span>
    エネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。
   </p>,
 <p>
    BREAK進化する前のゲッコウガの「ワザ・特性・弱点・抵抗力・にげる」を引きつぐ。
   </p>]

In [25]:
wazaCount = len(p) - len(h2) +1

h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
p = wazaPart.find_all('p')
# init
[ability, abilityDesc, spRule] = [''] * 3 
for area in h2:
    areaType = area.get_text().strip()
    if areaType == "特性":
        # ability
        print('learning an ability')
        ability = skills[0].get_text().strip()
        abilityDesc = p[0].get_text().strip()
        del skills[0]
        
    elif areaType == "特別なルール":
        # special rule
        print('learning a special rule')
        spRule = p[-1].get_text().strip()
        del p[-1]
        
    elif areaType == "GXワザ":
        # GX waza
        [GXCost, GXName, GXDamage, GXDesc] = [''] * 4
        print('learning a GX attack')
        GXCost = [span['class'][0].split('-')[1] for span in skills[-1].find_all('span', class_='icon')]
        GX = skills[-1].get_text().strip().split(' ')
        GXName = GX[0].strip()
        GXDamage = GX[-1]
        if not GXDamage[-2].isdigit():
            GXDamage = ''
        GXDesc = skills[-1].find_next_sibling('p')
        GXDesc = readEnergy(GXDesc).strip()
        del skills[-1], p[-2]
        
[wazaCount, skills, p]
# [GXCost, GXName, GXDamage, GXDesc]

learning an ability
learning a special rule


[0,
 [],
 [<p>
     自分の手札から
     <span class="icon-water icon">
  </span>
     エネルギーを1枚選び、トラッシュする。その後、相手のポケモンを1匹選び、ダメカンを6個のせる。この特性は、このポケモンがバトル場にいるなら、自分の番に1回使える。
    </p>]]

NameError: name 'wazaCost' is not defined

In [507]:
# init
[waza1Cost, waza1Name, waza1Damage, waza1Desc] = [''] * 4
[waza2Cost, waza2Name, waza2Damage, waza2Desc] = [''] * 4

waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
waza1 = skills[0].get_text().strip().split(' ')
waza1Name = waza1[0].strip()
waza1Damage = int(waza1[-1])
waza1Desc = skills[0].find_next_sibling('p').get_text().strip()

if len(skills) > 1:
    waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
    waza2 = skills[1].get_text().strip().split(' ')
    waza2Name = waza2[0].strip()
    waza2Damage = int(waza2[-1])
    waza2Desc = skills[1].find_next_sibling('p').get_text().strip()
[waza1Cost, waza1Name, waza1Damage, waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc]

[['none', 'none'],
 'しばりつける',
 50,
 '次の相手の番、このワザを受けたポケモンは、にげられない。',
 ['grass', 'grass'],
 'ジャングルライズ',
 100,
 'のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。その後、つけたポケモンのHPをすべて回復する。']

In [468]:
spRule

''

In [469]:
td = wazaPart.find_all('td')
td

[<td>
 <span class="icon-fire icon">
 </span>
      ×2
     </td>,
 <td>
      --
     </td>,
 <td class="escape">
 <span class="icon-none icon">
 </span>
 </td>]

In [472]:
[weakType, weakValue, resistType, resistValue] = [''] * 4

if td[0].find('span'):
    weakType = td[0].find('span')['class'][0].split('-')[1]
    weakValue = td[0].get_text().strip()

if td[1].find('span'):
    resistType = td[1].find('span')['class'][0].split('-')[1]
    resistValue = td[1].get_text().strip()
    
escape = len(td[2].find_all('span'))

[weakType, weakValue, resistType, resistValue, escape]

['fire', '×2', '', '', 1]

In [471]:
td[0].find('span')['class'][0].split('-')[1]

'fire'

In [56]:
# special engergy
twinEngergy = "https://www.pokemon-card.com/card-search/details.php/card/38482/regu/all"
card = readCard(twinEngergy)
card

<section class="Section">
<h1 class="Heading1 mt20">ツインエネルギー</h1>
<div class="Box">
<div class="LeftBox">
<img alt="ツインエネルギー" class="fit" src="/assets/images/card_images/large/S3a/038482_E_TSUINENERUGI.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 076 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
</div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">特殊エネルギー</h2>
<p>このカードは、ポケモンについているかぎり、【無】エネルギー2個ぶんとしてはたらく。<br/>
<br/>
「ポケモンV・GX」についているなら、【無】エネルギー1個ぶんとしてはたらく。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>

In [57]:
# supporter
onion = "https://www.pokemon-card.com/card-search/details.php/card/38477/regu/XY"
card = readCard(onion)
card

<section class="Section">
<h1 class="Heading1 mt20">オニオン</h1>
<div class="Box">
<div class="LeftBox">
<img alt="オニオン" class="fit" src="/assets/images/card_images/large/S3a/038477_T_ONION.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 071 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
<h4>イラストレーター</h4>
<a href="/card-search/index.php?regulation_illust=XY&amp;illust=take">take</a> </div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">サポート</h2>
<p>自分の山札を3枚引く。その後、自分の手札を3枚まで選び、トラッシュする。（必ず1枚はトラッシュする。）</p>
<p>サポートは、自分の番に1枚しか使えない。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>

In [ ]:
cardId = 38477 # start: 33001, end: 38493
readCard(cardId)

In [707]:
import sys
import pandas as pd

start = 38491
end = 38493

def scrapeCards(start, end):
    columns = ['cardId', 'cardType', 'name', 'img', 'regulation', 'setNum', 'setCount', 'rarity', 'dexNum',
                'dexClass', 'height', 'weight', 'dexDesc', 'author', 'desc', 'stage', 'hp', 'pType',
                'ability', 'abilityDesc', 'waza1Cost', 'waza1Name', 'waza1Damage',
                'waza1Desc', 'waza2Cost', 'waza2Name', 'waza2Damage', 'waza2Desc',
                'GXCost', 'GXName', 'GXDamage', 'GXDesc',
                'weakType', 'weakValue', 'resistType', 'resistValue', 'escape', 'spRule']

    df = pd.DataFrame(columns=columns)
    
    n = end - start+1
    for i in range(n):
        df.loc[i] = readCard(i+start)
        j = (i + 1) / n
        sys.stdout.write('\r')
        # the exact output you're looking for:
        sys.stdout.write("[%-20s] %d%%" % ('='*int(20*j), 100*j))
        sys.stdout.flush()

    df.to_csv(f'cards_jp_{start}_{end}.csv')

In [708]:
scrapeCards(start, end)

[====================] 100%

In [685]:
df.loc[2] = readCard(38493)
df

,cardId,cardType,name,img,regulation,setNum,setCount,rarity,dexNum,dexClass,...,waza2Cost,waza2Name,waza2Damage,waza2Desc,weakType,weakValue,resistType,resistValue,escape,spRule
0,38491,サポート,ネズ,/assets/images/card_images/large/S3/038491_T_N...,,109,100,sr_c,,,...,,,,,,,,,,
1,38492,サポート,ローズ,/assets/images/card_images/large/S3/038492_T_R...,,110,100,sr_c,,,...,,,,,,,,,,
2,38493,pokemon,ザルードV,/assets/images/card_images/large/S-P/038493_P_...,,108,S-P,,,,...,"[grass, grass]",ジャングルライズ,100,のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。...,fire,×2,,,1,ポケモンVがきぜつしたとき、相手はサイドを2枚とる。


In [704]:
import sys

n = 21
for i in range(n):
    j = (i + 1) / n
    sys.stdout.write('\r')
    # the exact output you're looking for:
    sys.stdout.write("[%-20s] %d%%" % ('='*int(20*j), 100*j))
    sys.stdout.flush()

[====================] 100%

In [706]:
print("[%-20s]" % '0')

[0                   ]
